In [ ]:
# ===== NB14 | CELL 1: header =====
from google.colab import drive
drive.mount('/content/drive')
print('CELL 1 DONE')

In [ ]:
# ===== NB14 | CELL 2: FINAL evaluation on the internal and external test sets =====
# WHAT: apply the frozen system (runs and thresholds from notebook 13) to both test sets and report per-label
#       precision, recall, F1 and AP with 95% bootstrap intervals, confusion matrices and PR curves.
# WHY:  this is the honest final score. Nothing is tuned here. The external test (Kaggle) is separate data,
#       so a drop from internal to external shows label noise and distribution shift.
import sys, importlib, json, os
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
sys.path.insert(0, f'{PROJECT_DIR}/src')
import toxlib, evallib
for m in (toxlib, evallib): importlib.reload(m)
from toxlib import LABELS, SPLITS, load_split, load_probs, macro_ap, tune_thresholds, per_label_report
from sklearn.metrics import precision_recall_curve
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
for d in ['graphs', 'tables', 'final']:
    os.makedirs(f'{PROJECT_DIR}/results/{d}', exist_ok=True)

cfg = json.load(open(f'{PROJECT_DIR}/results/final/final_config.json'))
runs, thr = cfg['runs'], cfg['thresholds']
print('Frozen system:', runs, '| frozen at', cfg['frozen_at'])
for split in SPLITS:                                           # save the final system's probabilities for later notebooks
    np.save(f'{PROJECT_DIR}/results/final/final_{split}.npy', evallib.avg_probs(PROJECT_DIR, runs, split))

summary, results = [], {}
for name, split in [('internal', 'test_internal'), ('external', 'test_external')]:
    Y = load_split(PROJECT_DIR, split)[LABELS].values
    Pf = np.load(f'{PROJECT_DIR}/results/final/final_{split}.npy')
    tab, macro = evallib.bootstrap_metrics(Y, Pf, thr, n_boot=500)
    results[name] = (Y, Pf)
    print(f'\n===== T7 FINAL {name.upper()} TEST ({len(Y)} comments) ====='); print(tab.to_string(index=False))
    print('Macro F1:', macro['macro_F1'], macro['macro_F1_95CI'], '| Macro AP:', macro['macro_AP'], macro['macro_AP_95CI'])
    tab.to_csv(f'{PROJECT_DIR}/results/tables/T7_final_{name}.csv', index=False)
    summary.append({'test set': name, 'rows': len(Y), **macro})
    # F12 confusion matrices (one binary matrix per label)
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for i, (l, a) in enumerate(zip(LABELS, axes.ravel())):
        pred = Pf[:, i] >= thr[l]; y = Y[:, i]
        cm = np.array([[((y == 0) & ~pred).sum(), ((y == 0) & pred).sum()], [((y == 1) & ~pred).sum(), ((y == 1) & pred).sum()]])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=a, xticklabels=['pred 0', 'pred 1'], yticklabels=['true 0', 'true 1'])
        a.set_title(l)
    plt.suptitle(f'F12: confusion matrices, {name} test'); plt.tight_layout()
    plt.savefig(f'{PROJECT_DIR}/results/graphs/F12_confusion_{name}.png', dpi=150); plt.show()
    # F13 PR curves
    plt.figure(figsize=(7, 5))
    for i, l in enumerate(LABELS):
        p, r, _ = precision_recall_curve(Y[:, i], Pf[:, i]); plt.plot(r, p, label=l)
    plt.xlabel('recall'); plt.ylabel('precision'); plt.title(f'F13: precision-recall curves, {name} test'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(f'{PROJECT_DIR}/results/graphs/F13_pr_curves_{name}.png', dpi=150); plt.show()

S = pd.DataFrame(summary); S.to_csv(f'{PROJECT_DIR}/results/tables/T7_summary.csv', index=False)
print('\nT7 summary:'); print(S.to_string(index=False))
# F14 macro scores with intervals
fig, ax = plt.subplots(figsize=(6, 4)); w = 0.35
for k, (col, lab) in enumerate([('macro_F1', 'Macro F1'), ('macro_AP', 'Macro AP')]):
    vals = S[col].values; err = np.array([[v - c[0], c[1] - v] for v, c in zip(vals, S[col + '_95CI'])]).T
    ax.bar(np.arange(2) + k * w, vals, w, yerr=err, capsize=4, label=lab)
ax.set_xticks(np.arange(2) + w / 2); ax.set_xticklabels(S['test set']); ax.legend(); ax.set_title('F14: final scores with 95% intervals')
plt.tight_layout(); plt.savefig(f'{PROJECT_DIR}/results/graphs/F14_final_scores.png', dpi=150); plt.show()
print('CELL 2 DONE')

In [ ]:
# ===== NB14 | CELL 3: all systems on the test sets (comparison table) =====
# WHAT: score every run and the final ensemble on both test sets, each with its OWN validation thresholds.
# WHY:  the report needs one table showing baseline -> DistilBERT -> RoBERTa -> ensemble. This table does not
#       change the choice: the final system was frozen on validation data in notebook 13.
import sys, importlib, json, os
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
sys.path.insert(0, f'{PROJECT_DIR}/src')
import toxlib; importlib.reload(toxlib)
from toxlib import LABELS, load_split, load_probs, macro_ap, tune_thresholds, per_label_report
import numpy as np, pandas as pd

cfg = json.load(open(f'{PROJECT_DIR}/results/final/final_config.json'))
Yva = load_split(PROJECT_DIR, 'val')[LABELS].values
Yt = {n: load_split(PROJECT_DIR, s)[LABELS].values for n, s in [('internal', 'test_internal'), ('external', 'test_external')]}
reg = pd.read_csv(f'{PROJECT_DIR}/results/experiment_logs/registry.csv')
systems = {r: [r] for r in reg.run_id if os.path.exists(f'{PROJECT_DIR}/results/predictions/{r}_val.npy')}
systems['FINAL ensemble'] = cfg['runs']
rows = []
for name, rs in systems.items():
    avg = lambda split: np.mean([load_probs(PROJECT_DIR, r, split) for r in rs], axis=0)
    thr = tune_thresholds(Yva, avg('val'))                 # each system uses ITS OWN validation thresholds
    row = {'system': name, 'val Macro AP': round(macro_ap(Yva, avg('val')), 4)}
    for n, split in [('internal', 'test_internal'), ('external', 'test_external')]:
        Pt = avg(split)
        row[f'{n} Macro F1'] = round(per_label_report(Yt[n], Pt, thr).F1.mean(), 4)
        row[f'{n} Macro AP'] = round(macro_ap(Yt[n], Pt), 4)
    rows.append(row)
T8 = pd.DataFrame(rows).sort_values('val Macro AP', ascending=False)
T8.to_csv(f'{PROJECT_DIR}/results/tables/T8_all_models_test.csv', index=False)
print('T8 all systems on the test sets (for the comparison table; the choice was frozen on validation):')
print(T8.to_string(index=False))
print('CELL 3 DONE')